# Flip significance probability vs. sample size

The data loading mirrors `figure-reversal-risk.ipynb` (the notebook that produces
`figure-flip-probability.pdf`), so the pooled results here are exactly those shown
in that figure. We then test whether the probability of a numerically induced
significance flip is associated with study sample size, across **all** extracted
results (reported significant and non-significant alike).

In [11]:
import pandas as pd
import numpy as np
import rglob
from pathlib import Path
from scipy.stats import pearsonr

root_dir = Path.cwd().parent
data_dir = root_dir / "papers_data"

In [12]:
# Load per-result uncertainty tables (same pipeline as figure-reversal-risk.ipynb)
files = rglob.rglob(data_dir, "uncertainty.csv")

df = pd.DataFrame()
for file in files:
    x = pd.read_csv(file)
    x["paper"] = Path(file).parent.name
    df = pd.concat([df, x])

# Exclude papers without proper uncertainty data
df = df[~df["paper"].isin(["Liu_2012", "Current"])]
df["significant"] = df["significant"].astype(bool)

# P(conclusion reverses): for significant results -> P(becomes non-sig)
df["proba_flip"] = np.where(
    df["significant"],
    1 - df["proba_significant"],
    df["proba_significant"],
)

print(f"Total results:            {len(df)}")
print(f"  Reported significant:     {df['significant'].sum()}")
print(f"  Reported non-significant: {(~df['significant']).sum()}")

Total results:            707
  Reported significant:     198
  Reported non-significant: 509


## Correlation across all extracted results

We correlate the flip significance probability with the study sample size across
all extracted results, pooling reported-significant and reported-non-significant
findings.

In [20]:
data = df.dropna(subset=["sample_size", "proba_flip"]).copy()

r, p = pearsonr(data["sample_size"], data["proba_flip"])

print(f"N (all extracted results): {len(data)}")
print(f"Pearson correlation between flip significance probability and sample size:")
print(f"  r = {r:.3f}")
print(f"  p = {p:.7f}")

N (all extracted results): 707
Pearson correlation between flip significance probability and sample size:
  r = -0.118
  p = 0.0016487


## Scatter plot

Flip significance probability against study sample size, with an ordinary
least-squares fit. Reported-significant and reported-non-significant results are
distinguished by colour.

In [14]:
import plotly.graph_objects as go

FONT, SZ_TICK, SZ_LABEL = "Helvetica", 11, 13
COL_SIG, COL_NONSIG, COL_FIT = "#c0392b", "#2c6fbf", "#111111"

x = data["sample_size"].to_numpy(dtype=float)
y = data["proba_flip"].to_numpy(dtype=float)

# OLS fit for the trend line
slope, intercept = np.polyfit(x, y, 1)
xs = np.array([x.min(), x.max()])
ys = slope * xs + intercept

fig = go.Figure()
for label, colour in [(True, COL_SIG), (False, COL_NONSIG)]:
    m = data["significant"] == label
    fig.add_trace(
        go.Scatter(
            x=x[m.to_numpy()],
            y=y[m.to_numpy()],
            mode="markers",
            name=("Reported significant" if label else "Reported non-significant"),
            marker=dict(color=colour, size=5, opacity=0.55, line=dict(width=0)),
            hovertemplate="sample size=%{x}<br>P(flip)=%{y:.3f}<extra></extra>",
        )
    )
fig.add_trace(
    go.Scatter(
        x=xs,
        y=ys,
        mode="lines",
        name="OLS fit",
        line=dict(color=COL_FIT, width=2, dash="dash"),
        hoverinfo="skip",
    )
)

fig.add_annotation(
    x=0.98,
    y=0.98,
    xref="paper",
    yref="paper",
    showarrow=False,
    text=f"Pearson r = {r:.3f}, p = {p:.3g} (n = {len(data)})",
    font=dict(family=FONT, size=SZ_LABEL),
    align="right",
    xanchor="right",
    yanchor="top",
)

axis = dict(
    showline=True,
    linewidth=1,
    linecolor="black",
    mirror=False,
    ticks="outside",
    tickfont=dict(family=FONT, size=SZ_TICK),
    gridcolor="rgba(0,0,0,0.08)",
)
fig.update_xaxes(
    title=dict(text="Sample size", font=dict(family=FONT, size=SZ_LABEL)), **axis
)
fig.update_yaxes(
    title=dict(
        text="Probability of significance flip", font=dict(family=FONT, size=SZ_LABEL)
    ),
    **axis,
)
fig.update_layout(
    width=720,
    height=460,
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family=FONT, size=SZ_TICK),
    legend=dict(
        x=0.98,
        y=0.88,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(0,0,0,0)",
        font=dict(family=FONT, size=SZ_TICK),
    ),
    margin=dict(l=70, r=20, t=20, b=60),
)
fig.show()

In [15]:
out_path = data_dir / "figure-flip-probability-vs-sample-size.pdf"
fig.write_image(str(out_path), scale=3, width=720, height=460)
print(f"Saved {out_path.name}")

Saved figure-flip-probability-vs-sample-size.pdf


In [21]:
# Sentence as it appears in the manuscript
print(
    "No statistical correlation was found between flip significance probability "
    f"and sample size (Pearson correlation r={r:.3f}, p={p:.7f})."
)

No statistical correlation was found between flip significance probability and sample size (Pearson correlation r=-0.118, p=0.0016487).


## Breakdown by statistical test

The extracted results use three test statistics: T-values, F-values and
correlation coefficients. We repeat the flip-probability / sample-size
correlation within each test type.

In [ ]:
TEST_NAMES = {"T": "T-values", "F": "F-values", "R": "Correlation coefficients"}

rows = []
for test in ["T", "F", "R"]:
    g = data[data["test"] == test]
    rt, pt = pearsonr(g["sample_size"], g["proba_flip"])
    rows.append(
        {
            "test": TEST_NAMES.get(test, test),
            "n": len(g),
            "pearson_r": round(rt, 3),
            "p_value": pt,
        }
    )

by_test = pd.DataFrame(rows)
by_test

,test,n,pearson_r,p_value
0,T-values,307,-0.095,9.586442e-02
1,F-values,204,-0.093,1.868587e-01
2,Correlation coefficients,196,-0.417,1.207370e-09


In [18]:
from plotly.subplots import make_subplots

order = ["T", "F", "R"]
fig2 = make_subplots(
    rows=1,
    cols=3,
    shared_yaxes=True,
    subplot_titles=[TEST_NAMES[t] for t in order],
    horizontal_spacing=0.04,
)

for j, test in enumerate(order, start=1):
    g = data[data["test"] == test]
    gx = g["sample_size"].to_numpy(dtype=float)
    gy = g["proba_flip"].to_numpy(dtype=float)
    rt, pt = pearsonr(gx, gy)
    for label, colour in [(True, COL_SIG), (False, COL_NONSIG)]:
        m = (g["significant"] == label).to_numpy()
        fig2.add_trace(
            go.Scatter(
                x=gx[m],
                y=gy[m],
                mode="markers",
                showlegend=(j == 1),
                name=("Reported significant" if label else "Reported non-significant"),
                marker=dict(color=colour, size=4, opacity=0.55, line=dict(width=0)),
                hovertemplate="sample size=%{x}<br>P(flip)=%{y:.3f}<extra></extra>",
            ),
            row=1,
            col=j,
        )
    s2, i2 = np.polyfit(gx, gy, 1)
    xs2 = np.array([gx.min(), gx.max()])
    fig2.add_trace(
        go.Scatter(
            x=xs2,
            y=s2 * xs2 + i2,
            mode="lines",
            showlegend=False,
            line=dict(color=COL_FIT, width=2, dash="dash"),
            hoverinfo="skip",
        ),
        row=1,
        col=j,
    )
    xref = "x domain" if j == 1 else f"x{j} domain"
    yref = "y domain" if j == 1 else f"y{j} domain"
    fig2.add_annotation(
        x=0.97,
        y=0.97,
        xref=xref,
        yref=yref,
        showarrow=False,
        text=f"r = {rt:.3f}<br>p = {pt:.3g}<br>n = {len(g)}",
        font=dict(family=FONT, size=SZ_TICK),
        align="right",
        xanchor="right",
        yanchor="top",
    )

fig2.update_xaxes(
    title=dict(text="Sample size", font=dict(family=FONT, size=SZ_LABEL)),
    showline=True,
    linewidth=1,
    linecolor="black",
    ticks="outside",
    tickfont=dict(family=FONT, size=SZ_TICK),
    gridcolor="rgba(0,0,0,0.08)",
)
fig2.update_yaxes(
    showline=True,
    linewidth=1,
    linecolor="black",
    ticks="outside",
    tickfont=dict(family=FONT, size=SZ_TICK),
    gridcolor="rgba(0,0,0,0.08)",
)
fig2.update_yaxes(
    title=dict(
        text="Probability of significance flip", font=dict(family=FONT, size=SZ_LABEL)
    ),
    row=1,
    col=1,
)
fig2.update_layout(
    width=1000,
    height=420,
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family=FONT, size=SZ_TICK),
    legend=dict(
        orientation="h",
        x=0.5,
        y=1.14,
        xanchor="center",
        bgcolor="rgba(0,0,0,0)",
        font=dict(family=FONT, size=SZ_TICK),
    ),
    margin=dict(l=70, r=20, t=70, b=60),
)
for ann in fig2.layout.annotations[:3]:
    ann.font = dict(family=FONT, size=SZ_LABEL)
fig2.show()

In [19]:
out_path2 = data_dir / "figure-flip-probability-vs-sample-size-by-test.pdf"
fig2.write_image(str(out_path2), scale=3, width=1000, height=420)
print(f"Saved {out_path2.name}")

Saved figure-flip-probability-vs-sample-size-by-test.pdf
